# 03 — MS-TCN-Style Teacher-Student Training on Breakfast

This notebook runs the first controlled training experiment for the updated project direction.

Core idea:

```text
Training:   text can be used
Inference:  only visual features are used
```

Models compared:

```text
1. baseline_visual_only      — visual-only TCN baseline
2. text_aware_teacher        — teacher using CLIP action text prototypes
3. student_ce_only           — video-only student trained without distillation
4. student_kd_video_only     — video-only student trained with KD from teacher
```

Default config is a scale-1 control run: 200 train videos, 50 test videos, 3 epochs.
For full experiments, change `RUN_MODE` below.


## 1. Runtime setup

This notebook can run locally on Linux or in Colab.

For the local Linux setup, it expects this repository structure:

```text
text-assisted-tas/
├── data/
│   ├── zenodo_ms_tcn_data/
│   │   └── breakfast/
│   │       ├── features/
│   │       ├── groundTruth/
│   │       ├── splits/
│   │       └── mapping.txt
│   └── text_assisted_tas/
│       └── breakfast/
│           └── text_embeddings/
│               ├── breakfast_clip_vitb16_text_embedding_config.json
│               ├── breakfast_clip_vitb16_text_embedding_metadata.csv
│               └── breakfast_clip_vitb16_text_embeddings.npy
├── notebooks/
└── runs/
```


In [86]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception:
    IN_COLAB = False
    print('Running outside Colab. Google Drive mount skipped.')


Running outside Colab. Google Drive mount skipped.


## 2. Imports and path configuration


In [87]:
from pathlib import Path
import os
import json
import random
import time
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


def find_project_root() -> Path:
    """Find the repository root for local runs or fall back to Colab Drive paths."""
    candidates = []

    env_root = os.environ.get('TAS_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))

    candidates.extend([
        Path.home() / 'text-assisted-tas',
        Path.home() / 'tas_project_local',
    ])

    for candidate in candidates:
        if (candidate / 'data' / 'zenodo_ms_tcn_data' / 'breakfast').exists():
            return candidate

    colab_root = Path('/content/drive/MyDrive/mmf_tas_lab_project')
    if colab_root.exists():
        return colab_root

    raise FileNotFoundError(
        'Could not find project root. Run the notebook from the repo, or set TAS_PROJECT_ROOT. '
        'Expected data/zenodo_ms_tcn_data/breakfast under the project root.'
    )


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data'

BREAKFAST_ROOT = DATA_ROOT / 'zenodo_ms_tcn_data' / 'breakfast'
TEXT_ASSISTED_ROOT = DATA_ROOT / 'text_assisted_tas' / 'breakfast'
TEXT_EMBEDDING_DIR = TEXT_ASSISTED_ROOT / 'text_embeddings'

RUNS_ROOT = PROJECT_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('BREAKFAST_ROOT:', BREAKFAST_ROOT)
print('TEXT_ASSISTED_ROOT:', TEXT_ASSISTED_ROOT)
print('RUNS_ROOT:', RUNS_ROOT)


IN_COLAB: False
PROJECT_ROOT: /home/mkirilin/text-assisted-tas
DATA_ROOT: /home/mkirilin/text-assisted-tas/data
BREAKFAST_ROOT: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast
TEXT_ASSISTED_ROOT: /home/mkirilin/text-assisted-tas/data/text_assisted_tas/breakfast
RUNS_ROOT: /home/mkirilin/text-assisted-tas/runs


## 3. Experiment configuration


In [88]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Options:
#   'local_debug'    -> very small local test
#   'scale1_control' -> current recommended controlled proof-of-concept
#   'full_split1'    -> full Breakfast split 1
RUN_MODE = 'full_split1'
SPLIT_ID = 1

if RUN_MODE == 'local_debug':
    RUN_NAME = 'mstcn_teacher_student_split1_local_debug'
    MAX_TRAIN_VIDEOS = 20
    MAX_TEST_VIDEOS = 10
    NUM_EPOCHS_BASELINE = 1
    NUM_EPOCHS_TEACHER = 1
    NUM_EPOCHS_STUDENT = 1
elif RUN_MODE == 'scale1_control':
    RUN_NAME = 'mstcn_teacher_student_split1_scale1_control'
    MAX_TRAIN_VIDEOS = 200
    MAX_TEST_VIDEOS = 50
    NUM_EPOCHS_BASELINE = 3
    NUM_EPOCHS_TEACHER = 3
    NUM_EPOCHS_STUDENT = 3
elif RUN_MODE == 'full_split1':
    RUN_NAME = 'mstcn_teacher_student_split1_full'
    MAX_TRAIN_VIDEOS = None
    MAX_TEST_VIDEOS = None
    NUM_EPOCHS_BASELINE = 10
    NUM_EPOCHS_TEACHER = 10
    NUM_EPOCHS_STUDENT = 10
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

BATCH_SIZE = 1
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

NUM_F_MAPS = 64
NUM_LAYERS = 8
NUM_STAGES = 2

KD_TEMPERATURE = 4.0
LAMBDA_CE = 1.0
LAMBDA_KD = 1.0

COPY_SELECTED_DATA_TO_LOCAL = False

RUN_ROOT = RUNS_ROOT / RUN_NAME
LOCAL_RUN_ROOT = RUN_ROOT
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('RUN_MODE:', RUN_MODE)
print('RUN_NAME:', RUN_NAME)
print('Split:', SPLIT_ID)
print('MAX_TRAIN_VIDEOS:', MAX_TRAIN_VIDEOS)
print('MAX_TEST_VIDEOS:', MAX_TEST_VIDEOS)
print('Epochs baseline/teacher/student:', NUM_EPOCHS_BASELINE, NUM_EPOCHS_TEACHER, NUM_EPOCHS_STUDENT)
print('COPY_SELECTED_DATA_TO_LOCAL:', COPY_SELECTED_DATA_TO_LOCAL)
print('RUN_ROOT:', RUN_ROOT)

if device != 'cuda':
    print('\nWARNING: CUDA is not available. The notebook will run on CPU and may be slow.')


Device: cpu
RUN_MODE: full_split1
RUN_NAME: mstcn_teacher_student_split1_full
Split: 1
MAX_TRAIN_VIDEOS: None
MAX_TEST_VIDEOS: None
Epochs baseline/teacher/student: 10 10 10
COPY_SELECTED_DATA_TO_LOCAL: False
RUN_ROOT: /home/mkirilin/text-assisted-tas/runs/mstcn_teacher_student_split1_full



## 4. Validate required files


In [89]:
required_paths = {
    'Breakfast features': BREAKFAST_ROOT / 'features',
    'Breakfast groundTruth': BREAKFAST_ROOT / 'groundTruth',
    'Breakfast mapping': BREAKFAST_ROOT / 'mapping.txt',
    'Breakfast splits': BREAKFAST_ROOT / 'splits',
    'Text embeddings': TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy',
    'Text embedding metadata': TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv',
}

for name, path in required_paths.items():
    print(f'{name}: {path} -> exists={path.exists()}')
    if not path.exists():
        raise FileNotFoundError(f'Missing required file/folder: {name}: {path}')

print('\nFile counts:')
print('features .npy:', len(list((BREAKFAST_ROOT / 'features').glob('*.npy'))))
print('groundTruth .txt:', len(list((BREAKFAST_ROOT / 'groundTruth').glob('*.txt'))))
print('split .bundle:', len(list((BREAKFAST_ROOT / 'splits').glob('*.bundle'))))


Breakfast features: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/features -> exists=True
Breakfast groundTruth: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/groundTruth -> exists=True
Breakfast mapping: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/mapping.txt -> exists=True
Breakfast splits: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/splits -> exists=True
Text embeddings: /home/mkirilin/text-assisted-tas/data/text_assisted_tas/breakfast/text_embeddings/breakfast_clip_vitb16_text_embeddings.npy -> exists=True
Text embedding metadata: /home/mkirilin/text-assisted-tas/data/text_assisted_tas/breakfast/text_embeddings/breakfast_clip_vitb16_text_embedding_metadata.csv -> exists=True

File counts:
features .npy: 1712
groundTruth .txt: 1712
split .bundle: 8


## 5. Load mapping, text embeddings, and splits


In [90]:
def read_lines(path: Path):
    return path.read_text().splitlines()


def load_mapping(mapping_path: Path):
    idx_to_label = {}
    label_to_idx = {}
    for line in read_lines(mapping_path):
        line = line.strip()
        if not line:
            continue
        idx, label = line.split(maxsplit=1)
        idx = int(idx)
        idx_to_label[idx] = label
        label_to_idx[label] = idx
    return idx_to_label, label_to_idx


def load_split(split_path: Path):
    video_ids = []
    for line in read_lines(split_path):
        line = line.strip()
        if line:
            video_ids.append(Path(line).stem)
    return video_ids


idx_to_label, label_to_idx = load_mapping(BREAKFAST_ROOT / 'mapping.txt')
num_classes = len(idx_to_label)

text_embeddings = np.load(TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embeddings.npy').astype(np.float32)
text_metadata = pd.read_csv(TEXT_EMBEDDING_DIR / 'breakfast_clip_vitb16_text_embedding_metadata.csv')

train_ids = load_split(BREAKFAST_ROOT / 'splits' / f'train.split{SPLIT_ID}.bundle')
test_ids = load_split(BREAKFAST_ROOT / 'splits' / f'test.split{SPLIT_ID}.bundle')

if MAX_TRAIN_VIDEOS is not None:
    train_ids = train_ids[:MAX_TRAIN_VIDEOS]
if MAX_TEST_VIDEOS is not None:
    test_ids = test_ids[:MAX_TEST_VIDEOS]

print('Number of classes:', num_classes)
print('Text embeddings shape:', text_embeddings.shape)
print('Train videos:', len(train_ids))
print('Test videos:', len(test_ids))
display(text_metadata.head())


Number of classes: 48
Text embeddings shape: (48, 512)
Train videos: 1460
Test videos: 252


,class_index,action_label,action_text,prompt
0,0,SIL,SIL,a video of the action: SIL
1,1,pour_cereals,pour cereals,a video of the action: pour cereals
2,2,pour_milk,pour milk,a video of the action: pour milk
3,3,stir_cereals,stir cereals,a video of the action: stir cereals
4,4,take_bowl,take bowl,a video of the action: take bowl


## 6. Optional local data cache


In [91]:
ACTIVE_FEATURE_DIR = BREAKFAST_ROOT / 'features'
ACTIVE_GT_DIR = BREAKFAST_ROOT / 'groundTruth'

if COPY_SELECTED_DATA_TO_LOCAL:
    selected_ids = sorted(set(train_ids + test_ids))
    LOCAL_DATA_CACHE = PROJECT_ROOT / '.cache' / 'breakfast_selected_cache' / RUN_NAME
    local_feature_dir = LOCAL_DATA_CACHE / 'features'
    local_gt_dir = LOCAL_DATA_CACHE / 'groundTruth'
    local_feature_dir.mkdir(parents=True, exist_ok=True)
    local_gt_dir.mkdir(parents=True, exist_ok=True)

    for video_id in tqdm(selected_ids, desc='Copying selected Breakfast files to local cache'):
        src_feature = BREAKFAST_ROOT / 'features' / f'{video_id}.npy'
        dst_feature = local_feature_dir / f'{video_id}.npy'
        if not dst_feature.exists():
            shutil.copy2(src_feature, dst_feature)

        src_gt = BREAKFAST_ROOT / 'groundTruth' / f'{video_id}.txt'
        dst_gt = local_gt_dir / f'{video_id}.txt'
        if not dst_gt.exists():
            shutil.copy2(src_gt, dst_gt)

    ACTIVE_FEATURE_DIR = local_feature_dir
    ACTIVE_GT_DIR = local_gt_dir

print('ACTIVE_FEATURE_DIR:', ACTIVE_FEATURE_DIR)
print('ACTIVE_GT_DIR:', ACTIVE_GT_DIR)


ACTIVE_FEATURE_DIR: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/features
ACTIVE_GT_DIR: /home/mkirilin/text-assisted-tas/data/zenodo_ms_tcn_data/breakfast/groundTruth


## 7. Dataset and DataLoader


In [92]:
class BreakfastTASDataset(Dataset):
    def __init__(self, video_ids, feature_dir: Path, gt_dir: Path, label_to_idx: dict):
        self.video_ids = list(video_ids)
        self.feature_dir = feature_dir
        self.gt_dir = gt_dir
        self.label_to_idx = label_to_idx

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, index):
        video_id = self.video_ids[index]
        feature_path = self.feature_dir / f'{video_id}.npy'
        gt_path = self.gt_dir / f'{video_id}.txt'

        features = np.load(feature_path).astype(np.float32)
        labels_str = read_lines(gt_path)
        labels = np.asarray([self.label_to_idx[x] for x in labels_str], dtype=np.int64)

        T = min(features.shape[1], len(labels))
        features = features[:, :T]
        labels = labels[:T]

        return {
            'video_id': video_id,
            'features': torch.from_numpy(features),
            'labels': torch.from_numpy(labels),
            'length': T,
        }


def tas_collate_fn(batch):
    batch_size = len(batch)
    feature_dim = batch[0]['features'].shape[0]
    max_len = max(item['length'] for item in batch)

    features = torch.zeros(batch_size, feature_dim, max_len, dtype=torch.float32)
    labels = torch.full((batch_size, max_len), fill_value=-100, dtype=torch.long)
    mask = torch.zeros(batch_size, 1, max_len, dtype=torch.float32)
    video_ids = []

    for i, item in enumerate(batch):
        T = item['length']
        features[i, :, :T] = item['features']
        labels[i, :T] = item['labels']
        mask[i, :, :T] = 1.0
        video_ids.append(item['video_id'])

    return {
        'video_ids': video_ids,
        'features': features,
        'labels': labels,
        'mask': mask,
        'lengths': torch.tensor([item['length'] for item in batch], dtype=torch.long),
    }


train_dataset = BreakfastTASDataset(train_ids, ACTIVE_FEATURE_DIR, ACTIVE_GT_DIR, label_to_idx)
test_dataset = BreakfastTASDataset(test_ids, ACTIVE_FEATURE_DIR, ACTIVE_GT_DIR, label_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=tas_collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=tas_collate_fn, num_workers=0)

sample = next(iter(train_loader))
print('Sample features:', sample['features'].shape)
print('Sample labels:', sample['labels'].shape)
print('Sample mask:', sample['mask'].shape)
print('Sample video:', sample['video_ids'][0])


Sample features: torch.Size([1, 2048, 1249])
Sample labels: torch.Size([1, 1249])
Sample mask: torch.Size([1, 1, 1249])
Sample video: P48_cam01_P48_juice


## 8. MS-TCN-style model definitions


In [93]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, mask):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return (x + out) * mask


class SingleStageTCN(nn.Module):
    def __init__(self, num_layers, num_f_maps, dim, num_classes):
        super().__init__()
        self.conv_in = nn.Conv1d(dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(2 ** i, num_f_maps) for i in range(num_layers)])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x, mask):
        out = self.conv_in(x) * mask
        for layer in self.layers:
            out = layer(out, mask)
        return self.conv_out(out) * mask


class MultiStageTCN(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, dim, num_classes):
        super().__init__()
        self.stage1 = SingleStageTCN(num_layers, num_f_maps, dim, num_classes)
        self.stages = nn.ModuleList([
            SingleStageTCN(num_layers, num_f_maps, num_classes, num_classes)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x, mask):
        outputs = []
        out = self.stage1(x, mask)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask, mask)
            outputs.append(out)
        return torch.stack(outputs, dim=0)


## 9. Text-aware teacher model


In [94]:
class TextPrototypeSingleStageTCN(nn.Module):
    def __init__(self, num_layers, num_f_maps, dim, text_embeddings_np):
        super().__init__()
        text_embeddings_t = torch.from_numpy(text_embeddings_np).float()
        text_embeddings_t = F.normalize(text_embeddings_t, dim=1)
        self.register_buffer('text_embeddings', text_embeddings_t)  # [C, E]

        num_classes, text_dim = text_embeddings_t.shape
        self.conv_in = nn.Conv1d(dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(2 ** i, num_f_maps) for i in range(num_layers)])
        self.visual_to_text = nn.Conv1d(num_f_maps, text_dim, kernel_size=1)
        self.logit_scale = nn.Parameter(torch.tensor(10.0))

    def forward(self, x, mask):
        out = self.conv_in(x) * mask
        for layer in self.layers:
            out = layer(out, mask)
        projected = self.visual_to_text(out)  # [B, E, T]
        projected = F.normalize(projected, dim=1)
        logits = torch.einsum('bet,ce->bct', projected, self.text_embeddings)
        logits = logits * self.logit_scale.clamp(1.0, 100.0)
        return logits * mask


class TextPrototypeMultiStageTCN(nn.Module):
    def __init__(self, num_stages, num_layers, num_f_maps, dim, text_embeddings_np):
        super().__init__()
        num_classes = text_embeddings_np.shape[0]
        self.stage1 = TextPrototypeSingleStageTCN(num_layers, num_f_maps, dim, text_embeddings_np)
        self.stages = nn.ModuleList([
            TextPrototypeSingleStageTCN(num_layers, num_f_maps, num_classes, text_embeddings_np)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x, mask):
        outputs = []
        out = self.stage1(x, mask)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask, mask)
            outputs.append(out)
        return torch.stack(outputs, dim=0)


## 10. Losses and metrics


In [95]:
def mstcn_supervised_loss(outputs, labels):
    total_loss = 0.0
    for s in range(outputs.shape[0]):
        logits = outputs[s].transpose(1, 2).contiguous()
        total_loss = total_loss + F.cross_entropy(
            logits.view(-1, logits.shape[-1]),
            labels.view(-1),
            ignore_index=-100,
        )
    return total_loss / outputs.shape[0]


def kd_loss_student_teacher(student_outputs, teacher_outputs, labels, temperature=4.0, lambda_ce=1.0, lambda_kd=1.0):
    ce = mstcn_supervised_loss(student_outputs, labels)
    student_last = student_outputs[-1]
    teacher_last = teacher_outputs[-1].detach()
    valid = labels != -100

    s_logits = student_last.transpose(1, 2)[valid]
    t_logits = teacher_last.transpose(1, 2)[valid]

    log_p_student = F.log_softmax(s_logits / temperature, dim=-1)
    p_teacher = F.softmax(t_logits / temperature, dim=-1)
    kd = F.kl_div(log_p_student, p_teacher, reduction='batchmean') * (temperature ** 2)
    return lambda_ce * ce + lambda_kd * kd, ce.detach(), kd.detach()


@torch.no_grad()
def framewise_accuracy(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        outputs = model(features, mask)
        preds = torch.argmax(outputs[-1], dim=1)
        valid = labels != -100
        correct += (preds[valid] == labels[valid]).sum().item()
        total += valid.sum().item()
    return correct / max(total, 1)


## 11. Build models


In [96]:
feature_dim = sample['features'].shape[1]

baseline_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)
teacher_model = TextPrototypeMultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, text_embeddings).to(device)
student_ce_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)
student_kd_model = MultiStageTCN(NUM_STAGES, NUM_LAYERS, NUM_F_MAPS, feature_dim, num_classes).to(device)

print('Feature dim:', feature_dim)
print('Classes:', num_classes)
print('Baseline parameters:', sum(p.numel() for p in baseline_model.parameters()))
print('Teacher parameters:', sum(p.numel() for p in teacher_model.parameters()))
print('Student CE-only parameters:', sum(p.numel() for p in student_ce_model.parameters()))
print('Student KD parameters:', sum(p.numel() for p in student_kd_model.parameters()))


Feature dim: 2048
Classes: 48
Baseline parameters: 404704
Teacher parameters: 465026
Student CE-only parameters: 404704
Student KD parameters: 404704


## 12. Training loops


In [97]:
def train_supervised_model(model, loader, optimizer, num_epochs, device, model_name):
    history = []
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_losses = []
        start_time = time.time()

        for batch in tqdm(loader, desc=f'{model_name} epoch {epoch}/{num_epochs}'):
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(features, mask)
            loss = mstcn_supervised_loss(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.item()))

        row = {
            'model': model_name,
            'epoch': epoch,
            'loss': float(np.mean(epoch_losses)) if epoch_losses else float('nan'),
            'train_acc': framewise_accuracy(model, train_loader, device),
            'test_acc': framewise_accuracy(model, test_loader, device),
            'seconds': time.time() - start_time,
        }
        history.append(row)
        print(row)
    return pd.DataFrame(history)


def train_student_with_kd(student, teacher, loader, optimizer, num_epochs, device):
    history = []
    teacher.eval()
    for epoch in range(1, num_epochs + 1):
        student.train()
        epoch_losses, epoch_ce, epoch_kd = [], [], []
        start_time = time.time()

        for batch in tqdm(loader, desc=f'student KD epoch {epoch}/{num_epochs}'):
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            with torch.no_grad():
                teacher_outputs = teacher(features, mask)
            student_outputs = student(features, mask)
            loss, ce, kd = kd_loss_student_teacher(
                student_outputs,
                teacher_outputs,
                labels,
                temperature=KD_TEMPERATURE,
                lambda_ce=LAMBDA_CE,
                lambda_kd=LAMBDA_KD,
            )
            loss.backward()
            optimizer.step()

            epoch_losses.append(float(loss.item()))
            epoch_ce.append(float(ce.item()))
            epoch_kd.append(float(kd.item()))

        row = {
            'model': 'student_kd_video_only',
            'epoch': epoch,
            'loss': float(np.mean(epoch_losses)) if epoch_losses else float('nan'),
            'ce': float(np.mean(epoch_ce)) if epoch_ce else float('nan'),
            'kd': float(np.mean(epoch_kd)) if epoch_kd else float('nan'),
            'train_acc': framewise_accuracy(student, train_loader, device),
            'test_acc': framewise_accuracy(student, test_loader, device),
            'seconds': time.time() - start_time,
        }
        history.append(row)
        print(row)
    return pd.DataFrame(history)


## 13. Train models


In [98]:
def save_single_model_checkpoint(model_name, model, history_df):
    path = RUN_ROOT / f'{model_name}_checkpoint.pt'
    payload = {
        'model_name': model_name,
        'run_mode': RUN_MODE,
        'run_name': RUN_NAME,
        'split_id': SPLIT_ID,
        'state_dict': model.state_dict(),
        'history': history_df.to_dict(orient='records'),
        'config': {
            'max_train_videos': MAX_TRAIN_VIDEOS,
            'max_test_videos': MAX_TEST_VIDEOS,
            'num_classes': num_classes,
            'feature_dim': feature_dim,
            'num_stages': NUM_STAGES,
            'num_layers': NUM_LAYERS,
            'num_f_maps': NUM_F_MAPS,
        },
    }
    torch.save(payload, path, _use_new_zipfile_serialization=False)
    print(f'Saved checkpoint: {path}')


baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
baseline_history = train_supervised_model(
    baseline_model, train_loader, baseline_optimizer, NUM_EPOCHS_BASELINE, device, 'baseline_visual_only'
)
display(baseline_history)
save_single_model_checkpoint('baseline_visual_only', baseline_model, baseline_history)

teacher_optimizer = torch.optim.Adam(teacher_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
teacher_history = train_supervised_model(
    teacher_model, train_loader, teacher_optimizer, NUM_EPOCHS_TEACHER, device, 'text_aware_teacher'
)
display(teacher_history)
save_single_model_checkpoint('text_aware_teacher', teacher_model, teacher_history)

student_ce_optimizer = torch.optim.Adam(student_ce_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
student_ce_history = train_supervised_model(
    student_ce_model, train_loader, student_ce_optimizer, NUM_EPOCHS_STUDENT, device, 'student_ce_only'
)
display(student_ce_history)
save_single_model_checkpoint('student_ce_only', student_ce_model, student_ce_history)

student_kd_optimizer = torch.optim.Adam(student_kd_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
student_kd_history = train_student_with_kd(
    student_kd_model, teacher_model, train_loader, student_kd_optimizer, NUM_EPOCHS_STUDENT, device
)
display(student_kd_history)
save_single_model_checkpoint('student_kd_video_only', student_kd_model, student_kd_history)


baseline_visual_only epoch 1/10: 100%|██████████| 1460/1460 [07:42<00:00,  3.16it/s]


{'model': 'baseline_visual_only', 'epoch': 1, 'loss': 2.6819297861971267, 'train_acc': 0.31748543256034645, 'test_acc': 0.3144639528948087, 'seconds': 631.1980998516083}


baseline_visual_only epoch 2/10: 100%|██████████| 1460/1460 [07:46<00:00,  3.13it/s]


{'model': 'baseline_visual_only', 'epoch': 2, 'loss': 1.9177095300122484, 'train_acc': 0.3431586104838895, 'test_acc': 0.3390414346823051, 'seconds': 673.4216916561127}


baseline_visual_only epoch 3/10: 100%|██████████| 1460/1460 [08:16<00:00,  2.94it/s]


...

{'model': 'student_kd_video_only', 'epoch': 9, 'loss': 1.2869717127452158, 'ce': 1.0686161010436817, 'kd': 0.21835561226287933, 'train_acc': 0.5757634881089698, 'test_acc': 0.4755273810795731, 'seconds': 468.140291929245}


student KD epoch 10/10: 100%|██████████| 1460/1460 [06:01<00:00,  4.04it/s]


{'model': 'student_kd_video_only', 'epoch': 10, 'loss': 1.2161223082305634, 'ce': 1.0067427009037913, 'kd': 0.20937960658171406, 'train_acc': 0.5906444935418413, 'test_acc': 0.4836136931118946, 'seconds': 467.2203803062439}


,model,epoch,loss,ce,kd,train_acc,test_acc,seconds
0,student_kd_video_only,1,3.651412,2.744683,0.906729,0.269313,0.299979,466.324080
1,student_kd_video_only,2,2.487485,1.972207,0.515278,0.408037,0.382751,486.401995
2,student_kd_video_only,3,2.070893,1.690884,0.380009,0.451511,0.409145,495.225612
3,student_kd_video_only,4,1.810563,1.502744,0.307819,0.518094,0.453831,477.501505
4,student_kd_video_only,5,1.656143,1.382717,0.273426,0.557932,0.477177,475.987537
5,student_kd_video_only,6,1.500524,1.258974,0.241550,0.490758,0.415105,476.171103
6,student_kd_video_only,7,1.396617,1.174657,0.221960,0.593724,0.472716,481.522774
7,student_kd_video_only,8,1.327912,1.108565,0.219347,0.605101,0.510455,475.233449
8,student_kd_video_only,9,1.286972,1.068616,0.218356,0.575763,0.475527,468.140292
9,student_kd_video_only,10,1.216122,1.006743,0.209380,0.590644,0.483614,467.220380


Saved checkpoint: /home/mkirilin/text-assisted-tas/runs/mstcn_teacher_student_split1_full/student_kd_video_only_checkpoint.pt


In [99]:
for name in [
    "baseline_model",
    "teacher_model",
    "student_ce_model",
    "student_kd_model",
    "train_loader",
    "test_loader",
    "device"
]:
    print(name, name in globals())

baseline_model True
teacher_model True
student_ce_model True
student_kd_model True
train_loader True
test_loader True
device True


## 14. Compare and save results


In [100]:
comparison_rows = []
for name, model in [
    ('baseline_visual_only', baseline_model),
    ('text_aware_teacher', teacher_model),
    ('student_ce_only', student_ce_model),
    ('student_kd_video_only', student_kd_model),
]:
    comparison_rows.append({
        'model': name,
        'train_acc': framewise_accuracy(model, train_loader, device),
        'test_acc': framewise_accuracy(model, test_loader, device),
        'uses_text_at_training': name in ['text_aware_teacher', 'student_kd_video_only'],
        'uses_text_at_inference': name == 'text_aware_teacher',
    })

df_comparison = pd.DataFrame(comparison_rows)
display(df_comparison)

comparison_path = RUN_ROOT / 'comparison.csv'
df_comparison.to_csv(comparison_path, index=False)
print('Saved comparison:', comparison_path)


,model,train_acc,test_acc,uses_text_at_training,uses_text_at_inference
0,baseline_visual_only,0.666812,0.533042,False,False
1,text_aware_teacher,0.440471,0.404254,True,True
2,student_ce_only,0.705830,0.543528,False,False
3,student_kd_video_only,0.590644,0.483614,True,False


Saved comparison: /home/mkirilin/text-assisted-tas/runs/mstcn_teacher_student_split1_full/comparison.csv


## 15. Save checkpoint and summary


In [101]:
checkpoint = {
    'config': {
        'run_mode': RUN_MODE,
        'run_name': RUN_NAME,
        'dataset': 'breakfast',
        'split_id': SPLIT_ID,
        'max_train_videos': MAX_TRAIN_VIDEOS,
        'max_test_videos': MAX_TEST_VIDEOS,
        'num_classes': num_classes,
        'feature_dim': feature_dim,
        'num_stages': NUM_STAGES,
        'num_layers': NUM_LAYERS,
        'num_f_maps': NUM_F_MAPS,
        'kd_temperature': KD_TEMPERATURE,
        'lambda_ce': LAMBDA_CE,
        'lambda_kd': LAMBDA_KD,
    },
    'idx_to_label': idx_to_label,
    'baseline_state_dict': baseline_model.state_dict(),
    'teacher_state_dict': teacher_model.state_dict(),
    'student_ce_state_dict': student_ce_model.state_dict(),
    'student_kd_state_dict': student_kd_model.state_dict(),
    'baseline_history': baseline_history.to_dict(orient='records'),
    'teacher_history': teacher_history.to_dict(orient='records'),
    'student_ce_history': student_ce_history.to_dict(orient='records'),
    'student_kd_history': student_kd_history.to_dict(orient='records'),
    'comparison': df_comparison.to_dict(orient='records'),
}

ckpt_path = RUN_ROOT / 'teacher_student_checkpoint.pt'
torch.save(checkpoint, ckpt_path, _use_new_zipfile_serialization=False)
print('Saved combined checkpoint:', ckpt_path)

summary = {
    'status': 'completed_run',
    'run_mode': RUN_MODE,
    'run_name': RUN_NAME,
    'dataset': 'Breakfast',
    'split': SPLIT_ID,
    'train_videos': len(train_ids),
    'test_videos': len(test_ids),
    'classes': num_classes,
    'feature_dim': feature_dim,
    'text_embedding_shape': list(text_embeddings.shape),
    'models': {
        'baseline_visual_only': 'visual-only MS-TCN-style model',
        'text_aware_teacher': 'teacher with CLIP action text prototypes',
        'student_ce_only': 'video-only student trained with cross entropy only',
        'student_kd_video_only': 'video-only student trained with KD from teacher',
    },
    'comparison': df_comparison.to_dict(orient='records'),
}

summary_path = RUN_ROOT / 'experiment_summary.json'
with summary_path.open('w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print('Saved summary:', summary_path)

print('\n03_mstcn_teacher_student_training_LOCAL completed.')


Saved combined checkpoint: /home/mkirilin/text-assisted-tas/runs/mstcn_teacher_student_split1_full/teacher_student_checkpoint.pt
{
  "status": "completed_run",
  "run_mode": "full_split1",
  "run_name": "mstcn_teacher_student_split1_full",
  "dataset": "Breakfast",
  "split": 1,
  "train_videos": 1460,
  "test_videos": 252,
  "classes": 48,
  "feature_dim": 2048,
  "text_embedding_shape": [
    48,
    512
  ],
  "models": {
    "baseline_visual_only": "visual-only MS-TCN-style model",
    "text_aware_teacher": "teacher with CLIP action text prototypes",
    "student_ce_only": "video-only student trained with cross entropy only",
    "student_kd_video_only": "video-only student trained with KD from teacher"
  },
  "comparison": [
    {
      "model": "baseline_visual_only",
      "train_acc": 0.666812295149178,
      "test_acc": 0.5330416958502004,
      "uses_text_at_training": false,
      "uses_text_at_inference": false
    },
    {
      "model": "text_aware_teacher",
      "train_

## 16. Next steps after this notebook

After the scale-1 control run is complete:

1. Add TAS metrics: Edit, F1@10, F1@25, F1@50;
2. Move the same pipeline to Assembly101;
3. Later compare against LTContext.
